In [ ]:
# =========================
# HARD ENV GATE
# =========================
import os, sys, subprocess, importlib, tempfile

TARGET_NUMPY    = "2.2.2"
TARGET_CATBOOST = "1.2.8"

SENTINEL = os.path.join(tempfile.gettempdir(), "ei_gate_restarted_once.txt")

def run(cmd):
    print(">>", " ".join(cmd))
    subprocess.check_call(cmd)

def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "--upgrade"] + list(pkgs))

def pip_uninstall(*pkgs):
    run([sys.executable, "-m", "pip", "uninstall", "-y"] + list(pkgs))

def need_version(mod, want):
    try:
        m = importlib.import_module(mod)
        return m.__version__ != want
    except Exception:
        return True

need_numpy    = need_version("numpy", TARGET_NUMPY)
need_catboost = need_version("catboost", TARGET_CATBOOST)

did_any_change = False

# 1) If anything mismatched — uninstall first (avoids ABI zombies)
if need_numpy or need_catboost:
    try:
        pip_uninstall("catboost", "numpy")
    except Exception as e:
        print("Uninstall note:", e)

# 2) Install exact versions (no quiet)
if need_numpy:
    pip_install(f"numpy=={TARGET_NUMPY}")
    did_any_change = True

if need_catboost:
    pip_install(f"catboost=={TARGET_CATBOOST}")
    did_any_change = True

# 3) Validate imports early
import numpy as _np
print("NumPy:", _np.__version__)
try:
    import catboost
    print("CatBoost:", catboost.__version__)
except Exception as e:
    print("CatBoost import failed:", repr(e))
    raise

# 4) Single safe restart (Jupyter/Colab aware)
if did_any_change and not os.path.exists(SENTINEL):
    with open(SENTINEL, "w") as f:
        f.write("1")
    print("🔁 Restarting kernel once to load fresh binaries...")

    # Prefer Jupyter-native shutdown if available
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and hasattr(ip, "kernel"):
            ip.kernel.do_shutdown(restart=True)
        else:
            raise RuntimeError("No IPython kernel")
    except Exception:
        # Fallback: hard kill (works in Colab/Linux)
        try:
            os.kill(os.getpid(), 9)
        except Exception:
            # Last resort: exit and ask user to restart
            sys.exit(0)

# If here: either no change or restart already done
# =========================

# =========================
# ===== RUN UI (no training) — single "Overview" with drivers, grouped recs, cumulative what-if
# =========================
import os, re, json, warnings
import numpy as np
import pandas as pd
import gradio as gr
import joblib

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
ART_DRIVE_URLS = {
    "models_all":   "https://drive.google.com/file/d/1ua4vxIrTyuT3Bub9ZXe6e6zJSdWDo6dK/view?usp=drive_link",
    "models_by_cat":"https://drive.google.com/file/d/11uKs4pLYXSmL0NbeyduRmXSxXMTwMBsn/view?usp=drive_link",
    "nmf":          "https://drive.google.com/file/d/1GJfGoTvy6Q4n0SosBNvT_tDrld_TLh7A/view?usp=drive_link",
    "meta":         "https://drive.google.com/file/d/1ZsS-38KkUdtgVM9FpMUXV_FLm4M48la0/view?usp=drive_link",
}
DATA_CSV_URL   = "https://drive.google.com/file/d/1TUEKqTs8cSnWcf1_nk8W1fJXqKIB3xJ-/view?usp=sharing"
DATA_CSV_LOCAL = "googleplay_merged.csv"

# sanity-check
_required = {"models_all","models_by_cat","nmf","meta"}
assert isinstance(ART_DRIVE_URLS, dict) and _required.issubset(ART_DRIVE_URLS), \
    "ART_DRIVE_URLS must be a dict with keys: models_all, models_by_cat, nmf, meta"
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

ART_DIR = "ei_artifacts"
EPS = 1e-9

# ---------- Drive fetch ----------
def _drive_file_id(url: str) -> str:
    m = re.search(r"/d/([^/]+)/", url or "")
    if not m: raise ValueError("Ожидаю ссылку вида .../file/d/<ID>/view")
    return m.group(1)

def fetch_from_drive(url: str, out_path: str) -> str:
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return out_path
    import gdown
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    file_id = _drive_file_id(url)
    uc = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(uc, out_path, quiet=False)
    if not (os.path.exists(out_path) and os.path.getsize(out_path) > 0):
        raise IOError(f"Не удалось скачать {out_path}")
    return out_path

def ensure_artifacts_local():
    os.makedirs(ART_DIR, exist_ok=True)
    targets = {
        "models_all":    os.path.join(ART_DIR, "models_all.joblib"),
        "models_by_cat": os.path.join(ART_DIR, "models_by_cat.joblib"),
        "nmf":           os.path.join(ART_DIR, "nmf.joblib"),
        "meta":          os.path.join(ART_DIR, "meta.json"),
    }
    for key, path in targets.items():
        fetch_from_drive(ART_DRIVE_URLS[key], path)
    fetch_from_drive(DATA_CSV_URL, DATA_CSV_LOCAL)

ensure_artifacts_local()

# ---------- Load artifacts ----------
MODELS_ALL = joblib.load(os.path.join(ART_DIR, "models_all.joblib"))
MODELS_BY_CAT = joblib.load(os.path.join(ART_DIR, "models_by_cat.joblib"))
NMF = joblib.load(os.path.join(ART_DIR, "nmf.joblib"))
with open(os.path.join(ART_DIR, "meta.json"), "r", encoding="utf-8") as f:
    META = json.load(f)
FEAT_COLS = META["feat_cols"]
LATENT_LABELS = META.get("latent_labels", {})
NMF_SHIFT = META["nmf_shift"]

# ---------- Feature prep ----------
def add_logs(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for c in ["organic_traffic_total","paid_traffic","dau","revenue","rank"]:
        d[f"log_{c}"] = np.log(np.clip(d[c].astype(float), a_min=1.0, a_max=None))
    d["sRank"] = -d["log_rank"]  # «лучше» = больше
    return d

def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    cols = FEAT_COLS
    for c in cols:
        if c not in df.columns: df[c] = 0.0
    return df[cols].fillna(0.0)

def apply_nmf(df: pd.DataFrame) -> pd.DataFrame:
    feats = ["log_paid_traffic","log_organic_traffic_total","log_dau","log_revenue","sRank"]
    for c in feats:
        if c not in df.columns: df[c] = 0.0
    X = df[feats].fillna(0.0).to_numpy()
    Xp = np.maximum(X + NMF_SHIFT, 0.0)
    Z = NMF.transform(Xp)
    for i in range(Z.shape[1]):
        df[f"z{i+1}"] = Z[:, i]
    return df

def predict_quantiles(cat: str, X: pd.DataFrame):
    models = MODELS_BY_CAT.get(cat, MODELS_ALL)
    p25 = float(models["p25"].predict(X)[0])
    p50 = float(models["p50"].predict(X)[0])  # трактуем как EI_p50
    p75 = float(models["p75"].predict(X)[0])
    return p25, p50, p75, models["p50"], models["p25"]

def predict_ei_p50(cat: str, X: pd.DataFrame) -> float:
    _, p50, _, _, _ = predict_quantiles(cat, X)
    return float(p50)

# ---------- Latent factor naming from NMF.components_ ----------
_NMF_BASE = ["log_paid_traffic", "log_organic_traffic_total", "log_dau", "log_revenue", "sRank"]
_BASE_TO_READABLE = {
    "log_paid_traffic": "Платный трафик/закупка",
    "log_organic_traffic_total": "Органика/эффективность",
    "log_dau": "DAU",
    "log_revenue": "Монетизация (ARPDAU/доля платящих)",
    "sRank": "Видимость/Ранг",
}

def _compose_label(main_idx, second_idx=None):
    a = _BASE_TO_READABLE[_NMF_BASE[main_idx]]
    if second_idx is None:
        return a
    b = _BASE_TO_READABLE[_NMF_BASE[second_idx]]
    return f"{a} × {b}"

def infer_factor_labels(nmf_model) -> dict:
    try:
        W = nmf_model.components_  # (K, 5)
        labels = {}
        for k in range(W.shape[0]):
            row = W[k, :]
            order = np.argsort(row)[::-1]
            main_idx = int(order[0])
            second_idx = int(order[1]) if W[k, order[1]] >= 0.75 * W[k, order[0]] else None
            labels[f"z{k+1}"] = _compose_label(main_idx, second_idx)
        return labels
    except Exception:
        return {}

FACTOR_NAME_MAP = infer_factor_labels(NMF)

# ---------- Human-readable feature names ----------
NAME_MAP = {
    "sRank": "Ранг в чартах (меньше — лучше)",
    "log_dau": "DAU",
    "log_revenue": "Монетизация (ARPDAU/доля платящих)",
    "log_organic_traffic_total": "Органический трафик (ниже потенциала)",
}
def human_name(feat: str) -> str:
    if feat.startswith("z"):
        return FACTOR_NAME_MAP.get(feat, LATENT_LABELS.get(feat, "Латентный драйвер"))
    return NAME_MAP.get(feat, feat)

# ---------- SHAP ----------
def shap_topk(model, Xrow: pd.DataFrame, k: int = 5):
    from catboost import Pool
    sv = model.get_feature_importance(Pool(Xrow, label=[0.0]), type="ShapValues")
    vals = dict(zip(FEAT_COLS, sv[0, :-1].tolist()))
    deny = {
        "dlogP","age_days","is_update_day","days_since_update","k_update",
        "is_feature_day","days_since_feature","k_feature","log_paid_traffic"
    }
    items = [(f, v) for f, v in vals.items() if f not in deny]
    items.sort(key=lambda x: abs(x[1]), reverse=True)
    return items[:k]

# ---------- RAW: just to build category list ----------
RAW = pd.read_csv(DATA_CSV_LOCAL, low_memory=False)
if "Category" in RAW.columns and "category" not in RAW.columns:
    RAW = RAW.rename(columns={"Category":"category"})
if "category" not in RAW.columns:
    RAW["category"] = "Unknown"
CATS = sorted([str(x) for x in RAW["category"].dropna().unique().tolist()])

# ---------- Gates & grouping ----------

GATES = {
    "dau_pct":   (10, 50),   # %
    "arpdau_pct":(5,  30),   # %
    "paid_pct":  (5,  40),   # %
    "rank_pos":  (0,  50),   # positions (min = 0 now)
}

def _scale_between(w: float, lo: float, hi: float) -> float:
    w = float(max(0.0, min(1.0, w)))
    return lo + (hi - lo) * w

def _class_for_feature(feat: str) -> str:
    """Отнести фичу (включая латенты) к actionable bucket."""
    if feat == "log_revenue": return "monetization"
    if feat == "sRank": return "rank"
    if feat == "log_dau": return "dau"
    if feat == "log_organic_traffic_total": return "organic_dep"  # не даём прямого действия

    if feat.startswith("z"):
        label = human_name(feat).lower()
        if "монетизац" in label: return "monetization"
        if "видимость" in label or "ранг" in label: return "rank"
        if "платный" in label or "закупка" in label: return "paid"
        if "dau" in label: return "dau"
        if "органика" in label: return "dau"  # ближайшее actionable
        return "dau"

    return "other"

def grouped_recommendations(drivers):
    """
    Объединяем схожие действия. Возвращаем (список рекомендаций, веса по бакетам).
    Веса считаем из |SHAP| и нормируем на сумму топ-k.
    """
    if not drivers:
        return [], {}

    weights = np.array([abs(v) for _, v in drivers], dtype=float)
    total = float(weights.sum()) if weights.sum() > 0 else 1.0
    norm = (weights / total).tolist()

    agg = {"dau": 0.0, "monetization": 0.0, "paid": 0.0, "rank": 0.0}
    for (feat, _), w in zip(drivers, norm):
        bucket = _class_for_feature(feat)
        if bucket in agg:
            agg[bucket] += w

    recs = []
    if agg["dau"] > 0:
        pct = int(round(_scale_between(agg["dau"], *GATES["dau_pct"])))
        recs.append(f"DAU: увеличить на ~{pct}%")
    if agg["monetization"] > 0:
        pct = int(round(_scale_between(agg["monetization"], *GATES["arpdau_pct"])))
        recs.append(f"Монетизация (ARPDAU/доля платящих): увеличить на ~{pct}%")
    if agg["paid"] > 0:
        pct = int(round(_scale_between(agg["paid"], *GATES["paid_pct"])))
        recs.append(f"Paid Traffic: увеличить на ~{pct}%")
    if agg["rank"] > 0:
        pos = int(round(_scale_between(agg["rank"], *GATES["rank_pos"])))
        recs.append(f"Ранг: улучшить на ~{pos} позиций (меньше — лучше)")

    return recs, agg

# ---------- helpers for what-if ----------
def _row_to_X(row):
    d = add_logs(pd.DataFrame([row]))
    # convenience flags for model compatibility
    for c, v in [("dlogP",0.0),("is_update_day",0),("days_since_update",-1),("k_update",0.0),
                 ("is_feature_day",0),("days_since_feature",-1),("k_feature",0.0)]:
        d[c] = v
    d = apply_nmf(d)
    return build_feature_matrix(d)

def _apply_grouped_deltas(row, buckets):
    """Apply grouped deltas to a copy of row; returns new row and textual summary of deltas."""
    new = dict(row)
    notes = []

    if buckets.get("dau", 0) > 0:
        pct = _scale_between(buckets["dau"], *GATES["dau_pct"]) / 100.0
        new["dau"] = new["dau"] * (1.0 + pct)
        notes.append(f"DAU × {(1.0 + pct):.2f}")

    if buckets.get("monetization", 0) > 0:
        pct = _scale_between(buckets["monetization"], *GATES["arpdau_pct"]) / 100.0
        new["revenue"] = max(1.0, new.get("revenue", 1.0)) * (1.0 + pct)
        notes.append(f"Монетизация × {(1.0 + pct):.2f}")

    if buckets.get("paid", 0) > 0:
        pct = _scale_between(buckets["paid"], *GATES["paid_pct"]) / 100.0
        new["paid_traffic"] = new["paid_traffic"] * (1.0 + pct)
        notes.append(f"Paid × {(1.0 + pct):.2f}")

    if buckets.get("rank", 0) > 0:
        pos = int(round(_scale_between(buckets["rank"], *GATES["rank_pos"])))
        new["rank"] = max(1.0, new["rank"] - pos)  # rank: меньше — лучше
        notes.append(f"Rank −{pos}")

    return new, ", ".join(notes) if notes else "—"

# ---------- Iterative what-if simulation (cumulative, until target or caps) ----------
def predict_ei_p50_local(category, row):
    return predict_ei_p50(category, _row_to_X(row))

def simulate_to_target_cumulative(category, start_row, target_org,
                                  damping=0.6, tol=0.02,
                                  hard_caps=None, safety_cap=200):
    """
    Крутит рекомендации пока:
      - органика (эвристика EI*Paid c демпфированием) не достигнет цели (±tol),
      - или не упрёмся в границы по всем метрикам,
      - или не сработает safety_cap.

    Возвращает dict с суммарными изменениями и финальными метриками.
    """
    caps = hard_caps or {
        "rank_min": 1.0,
        "dau_max_mult": 10.0,
        "paid_max_mult": 10.0,
        "revenue_max_mult": 10.0,
    }

    row = dict(start_row)

    cum = {
        "dau_mult": 1.0,
        "paid_mult": 1.0,
        "revenue_mult": 1.0,
        "rank_improved_positions": 0,
    }

    def _infer_org(r):
        ei = predict_ei_p50_local(category, r)
        return float(max(1.0, ei * max(1.0, r["paid_traffic"]))), ei

    org_inf, ei = _infer_org(row)
    org_est = float(row.get("organic_traffic_total") or org_inf)

    it = 0
    while it < safety_cap:
        it += 1
        if org_est >= target_org * (1.0 - tol):
            break

        try:
            _, _, _, model_p50, _ = predict_quantiles(category, _row_to_X(row))
            drivers = shap_topk(model_p50, _row_to_X(row), k=5)
        except Exception:
            drivers = []
        recs, buckets = grouped_recommendations(drivers)

        if not drivers or all((buckets.get(k,0.0) <= 1e-6) for k in ["dau","monetization","paid","rank"]):
            break

        # step deltas
        dau_pct  = _scale_between(buckets.get("dau", 0.0),           *GATES["dau_pct"]) / 100.0
        mon_pct  = _scale_between(buckets.get("monetization", 0.0),  *GATES["arpdau_pct"]) / 100.0
        paid_pct = _scale_between(buckets.get("paid", 0.0),          *GATES["paid_pct"]) / 100.0
        rank_pos = int(round(_scale_between(buckets.get("rank", 0.0), *GATES["rank_pos"])))

        # apply with caps
        if dau_pct > 0 and cum["dau_mult"] < caps["dau_max_mult"]:
            mult = min(caps["dau_max_mult"] / cum["dau_mult"], 1.0 + dau_pct)
            row["dau"] *= mult; cum["dau_mult"] *= mult

        if mon_pct > 0 and cum["revenue_mult"] < caps["revenue_max_mult"]:
            mult = min(caps["revenue_max_mult"] / cum["revenue_mult"], 1.0 + mon_pct)
            row["revenue"] = max(1.0, row.get("revenue", 1.0)) * mult
            cum["revenue_mult"] *= mult

        if paid_pct > 0 and cum["paid_mult"] < caps["paid_max_mult"]:
            mult = min(caps["paid_max_mult"] / cum["paid_mult"], 1.0 + paid_pct)
            row["paid_traffic"] *= mult; cum["paid_mult"] *= mult

        if rank_pos > 0 and row["rank"] > caps["rank_min"]:
            improve = min(rank_pos, int(round(row["rank"] - caps["rank_min"])))
            if improve > 0:
                row["rank"] -= improve
                cum["rank_improved_positions"] += improve

        org_inf_new, ei_new = _infer_org(row)
        org_est = (1 - damping) * org_inf_new + damping * org_est

        if (dau_pct <= 1e-6 and mon_pct <= 1e-6 and paid_pct <= 1e-6 and rank_pos <= 0):
            break
        if (cum["dau_mult"] >= caps["dau_max_mult"]
            and cum["revenue_mult"] >= caps["revenue_max_mult"]
            and cum["paid_mult"] >= caps["paid_max_mult"]
            and row["rank"] <= caps["rank_min"]):
            break

    res = {
        "iterations": it,
        "target": float(target_org),
        "final_org_est": float(org_est),
        "final_ei": float(ei_new) if 'ei_new' in locals() else float(ei),
        "final_paid": float(row["paid_traffic"]),
        "final_rank": float(row["rank"]),
        "cum_dau_pct": (cum["dau_mult"] - 1.0) * 100.0,
        "cum_monetization_pct": (cum["revenue_mult"] - 1.0) * 100.0,
        "cum_paid_pct": (cum["paid_mult"] - 1.0) * 100.0,
        "cum_rank_positions": int(cum["rank_improved_positions"]),
        "hit_caps": {
            "dau": cum["dau_mult"] >= caps["dau_max_mult"],
            "monetization": cum["revenue_mult"] >= caps["revenue_max_mult"],
            "paid": cum["paid_mult"] >= caps["paid_max_mult"],
            "rank": row["rank"] <= caps["rank_min"],
        },
        "reached": float(org_est) >= float(target_org) * (1.0 - tol),
    }
    return res

# ---------- UI callbacks ----------
def ui_drivers_and_recs(category, dau, paid, org, rank, target_org):
    row = {
        "category": str(category or "Unknown"),
        "date": pd.Timestamp("today").normalize(),
        "app_id": "probe",
        "dau": float(dau or 0),
        "paid_traffic": float(paid or 0),
        "organic_traffic_total": float(org or 0),
        "rank": float(rank or 500),
        "revenue": 1.0,
    }
    df = apply_nmf(add_logs(pd.DataFrame([row])))
    X = build_feature_matrix(df)

    _, p50, _, model_p50, _ = predict_quantiles(row["category"], X)
    org_inf = float(max(1.0, p50 * max(1.0, row["paid_traffic"])))

    # прогноз (сверху)
    forecast_md = (
        "### Эвристический прогноз\n"
        f"EI (p50) ≈ **{p50:.3f}**  \n"
        f"Текущая органика (введено): **{int(row['organic_traffic_total'])}**  \n"
        f"Оценка по модели (EI×Paid): **{int(round(org_inf))}**"
    )

    # приоритеты драйверов (ABS SHAP, без знаков)
    try:
        drivers = shap_topk(model_p50, X, k=5)
    except Exception:
        drivers = []
    if drivers:
        lines = ["### Приоритеты драйверов (SHAP)"]
        for f, v in drivers:
            lines.append(f"- **{human_name(f)}** — {abs(v):.3f}")
        drivers_md = "\n".join(lines)
    else:
        drivers_md = "### Приоритеты драйверов (SHAP)\n—"

    # рекомендации первого шага (объединённые)
    recs, _ = grouped_recommendations(drivers)
    if recs:
        recs_md = "### Рекомендации — Итерация 1\n" + "\n".join(f"- {r}" for r in recs)
    else:
        recs_md = "### Рекомендации — Итерация 1\n—"

    return forecast_md, drivers_md, recs_md

def ui_summary_to_target(category, dau, paid, org, rank, target_org):
    row = {
        "category": str(category or "Unknown"),
        "date": pd.Timestamp("today").normalize(),
        "app_id": "probe",
        "dau": float(dau or 0),
        "paid_traffic": float(paid or 0),
        "organic_traffic_total": float(org or 0),
        "rank": float(rank or 500),
        "revenue": 1.0,
    }
    target_org = float(target_org or 0)

    res = simulate_to_target_cumulative(
        row["category"], row, target_org,
        damping=0.6, tol=0.02, hard_caps=None, safety_cap=200
    )

    reached = res["reached"]
    iters   = res["iterations"]

    lines = ["### Сводная рекомендация\n"]
    lines.append(f"Цель органики: **{int(round(res['target']))}**")
    lines.append(f"Итоговая оценка органики: **{int(round(res['final_org_est']))}**")
    lines.append(f"Достигнута? {'Да ✅' if reached else 'Нет ❌'} (за {iters} итераций)")
    lines.append("")
    lines.append("**Суммарно от исходной точки:**")
    lines.append(f"- DAU: **+{int(round(max(0.0, res['cum_dau_pct'])))}%**")
    lines.append(f"- Монетизация (ARPDAU/доля платящих): **+{int(round(max(0.0, res['cum_monetization_pct'])))}%**")
    lines.append(f"- Paid Traffic: **+{int(round(max(0.0, res['cum_paid_pct'])))}%**")
    lines.append(f"- Ранг: улучшить на **{res['cum_rank_positions']}** позиций (до {int(res['final_rank'])})")
    lines.append("")
    lines.append("**Финальные метрики (для проверки):**")
    lines.append(f"- EI (p50) ≈ **{res['final_ei']:.3f}**")
    lines.append(f"- Paid ≈ **{int(round(res['final_paid']))}**")
    lines.append(f"- Rank ≈ **{int(round(res['final_rank']))}**")

    if not reached:
        caps = res["hit_caps"]
        reasons = []
        if caps["rank"]: reasons.append("ранг упёрся в 1")
        if caps["dau"]: reasons.append("DAU достиг предела симуляции")
        if caps["monetization"]: reasons.append("монетизация достигла предела симуляции")
        if caps["paid"]: reasons.append("paid достиг предела симуляции")
        lines.append("")
        lines.append("**Почему цель не достигнута:** " + ("; ".join(reasons) if reasons else "модель не видит дальнейших траекторий роста в текущем диапазоне признаков."))
        lines.append("_Примечание: расчёт эвристический (органика ≈ EI×Paid)._")

    return "\n".join(lines)

# ---------- UI ----------
CUSTOM_CSS = """
.gradio-container {max-width: 1180px !important}
.card {border: 1px solid #2a2a2a; border-radius: 14px; padding: 14px}
small {opacity:.85}
.dataframe th, .dataframe td { font-size: 12px; padding: 4px 6px; }

/* снимаем ограничение по высоте */
.output-scroll, .wrap.svelte-1ipelgc {
    max-height: none !important;
}
"""


with gr.Blocks(title="Organic Efficiency Scanner", css=CUSTOM_CSS) as app:
    gr.Markdown("## 🚦 Efficiency Scanner — индикатор\n"
                "_Артефакты и CSV подтягиваются с Google Drive._")

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group(elem_classes="card"):
                gr.Markdown("### Ввод")
                i_cat    = gr.Dropdown(choices=CATS, value=CATS[0] if CATS else None, label="Category")
                i_dau    = gr.Number(label="DAU (текущ.)", value=15000, precision=0)
                i_paid   = gr.Number(label="Paid traffic (текущ.)", value=600, precision=0)
                i_org    = gr.Number(label="Organic (текущ.)", value=200, precision=0)
                i_rank   = gr.Number(label="Rank (меньше — лучше)", value=250)
                i_target = gr.Number(label="Target organic (цель)", value=5000, precision=0)
                btn_calc = gr.Button("Рассчитать драйверы и рекомендации", variant="primary")
                btn_sum  = gr.Button("🧮 Сводная рекомендация", variant="secondary")

        with gr.Column(scale=1):
            with gr.Group(elem_classes="card"):
                o_forecast = gr.Markdown("—")  # Эвристический прогноз (сверху)
            with gr.Group(elem_classes="card"):
                o_drivers  = gr.Markdown("—")  # Приоритеты драйверов (SHAP)
            with gr.Group(elem_classes="card"):
                o_recs1    = gr.Markdown("—")  # Рекомендации — Итерация 1
            with gr.Group(elem_classes="card"):
                o_summary  = gr.Markdown("—")  # Сводная рекомендация

    # кнопки
    btn_calc.click(
        ui_drivers_and_recs,
        inputs=[i_cat, i_dau, i_paid, i_org, i_rank, i_target],
        outputs=[o_forecast, o_drivers, o_recs1]
    )
    btn_sum.click(
        ui_summary_to_target,
        inputs=[i_cat, i_dau, i_paid, i_org, i_rank, i_target],
        outputs=[o_summary]
    )

    app.launch(server_name="0.0.0.0", share=True)

